# 第3章　データセットの構築とキュレーション ― AI開発の基盤を作る**『医療診断支援AIの社会実装（社会実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-social

## 匿名化を手順に落とす ― DICOMタグと検証

In [ ]:
import pydicomds = pydicom.dcmread("in.dcm")for tag in ["PatientName","PatientID","OtherPatientIDs","ReferringPhysicianName",            "InstitutionName","OperatorsName","StudyDescription"]:    if tag in ds: setattr(ds, tag, "")if "PatientBirthDate" in ds:                           # 無い症例もあるので、存在を確認してから触る    age = compute_age(ds.PatientBirthDate, ds.StudyDate)   # 撮影日 − 生年月日 で年齢を出す    ds.PatientAge = f"{age:03d}Y"                      # 生年月日は年齢へ丸める    del ds.PatientBirthDate                            # 元の生年月日タグは削除# 日付は基準日からの相対日数へ、受付番号は除去for tag in ["StudyDate", "SeriesDate", "AcquisitionDate", "ContentDate"]:    if tag in ds: setattr(ds, tag, shift_date(getattr(ds, tag), base_date))ds.AccessionNumber = ""# UIDは辞書で一貫再発番（同一検査の紐付けを維持しつつ匿名化）。# Studyだけ変えても、SeriesとSOPが残れば元PACSの当該画像に逆引きできてしまう。for tag in ["StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID"]:    setattr(ds, tag, remap_uid(getattr(ds, tag)))ds.file_meta.MediaStorageSOPInstanceUID = ds.SOPInstanceUID   # file_meta側も揃えるds.remove_private_tags()                       # プライベートタグは原則除去ds.save_as("out.dcm")

## データ品質を、継続的に検査する ― 取り込み時の関所

In [ ]:
# 取り込み時のデータ品質検査（期待仕様に照らして合否を出す）def check(study, index):    ok  = study.modality in {"CT", "MR"}                    # 想定モダリティ    ok &= 0.3 <= study.pixel_spacing_mm <= 1.5              # 画素間隔が現実的範囲（数値は例示）    ok &= study.slices >= 20                                # スライス枚数の下限    ok &= not phi_present(study.tags)                       # 匿名化の取りこぼし検査    if study.modality == "CT":                              # 値域の検査はモダリティで分岐する        ok &= study.hu_converted                            # HUへ変換済み（変換係数あり）か        ok &= (-1100 <= study.hu_min) and (study.hu_max <= 3100)   # プロトコル別のHU基準（例示）    else:                                                   # MRにHUは無い。シーケンスに応じた信号強度・品質基準へ        ok &= mr_intensity_ok(study)    ok &= not is_near_duplicate(study, index)              # 重複・近重複でないか    return ok# 不合格は破棄せず「隔離」し、理由付きで記録（後で見直せるように）